# Qwen3.5-9B 高等数学解题助手 —— QLoRA 微调实验报告

## 摘要

本实验以 `Qwen/Qwen3.5-9B` 为基座，在单张 RTX 4090D（24GB）上使用 4-bit NF4 QLoRA 完成中文高等数学监督微调。实验依次开展 300 条冒烟测试、5k 训练、20k 训练和结构增强版 v3 训练，并在旧隔离集 iso300 与同源结构隔离集 eval_v3 上进行对照。结果表明：QLoRA 能以约 8.5–8.7 GiB 峰值显存稳定训练 9B 级模型，并将解题输出协议合规率提升到 100%；但同构数据从 5k 扩大到 20k 未改善 iso300（均为 37.7%）。v3 在 eval_v3 上由旧适配器的 93%–94%提升到 100%，同时在 iso300 上回落到 34.3%，说明定向数据能强化目标分布，但也会造成能力重分配。实验由此完成了 QLoRA 从环境、数据、训练、适配器推理到双基准评测和失败分析的完整学习闭环。

> **结论边界**：eval_v3 与训练集题面及结构签名零重合，但两者仍来自同一生成体系且 eval_v3 全为难度 2；100% 代表受控同源分布中的结构迁移，不代表开放世界高等数学正确率。

## 一、实验目的

**主目标**：用自建的中文高等数学 SFT 数据集，通过 QLoRA 对 `Qwen/Qwen3.5-9B` 进行参数高效微调，得到一个**擅长解答中文高等数学题**的专用智能助手——在保留基座通用能力的前提下，学会规范的解题推理格式。

### 1.1 能力目标

- 正确求解 29 类高等数学题型：极限、一元微分/积分、级数、多元微积分、微分方程、几何应用等；
- 输出规范的解题轨迹：先用 `<think>...</think>` 展示推导，再给出结论，最终答案统一放入 `\boxed{}`；
- 提升中文数学表达与 LaTeX 公式书写的规范性与稳定性。

### 1.2 工程与研究目标

- 验证「单卡 RTX 4090D (24GB) + QLoRA」训练 9.65B 模型的完整链路（环境已含 flash-attn 加速）；
- 建立**可复现的对照实验**：基座模型 vs 5k-LoRA vs 20k-LoRA，在同一隔离评测集上对比；
- 沉淀训练配置与评测流程，为后续 GRPO（利用 `canonical_answer` 可自动验算的题目）打基础。

### 1.3 成功标准（初版）

| 指标 | 目标 |
|------|------|
| 评测集答案正确率（`\boxed{}` 提取 + LaTeX 等价判定） | 相对基座显著提升（≥ +15pp 为佳） |
| 格式合规率（含 think 推理块且含 `\boxed{}`） | ≥ 95% |
| 泛化检查（未见模板/变体题面） | 无明显退化，不出现"背模板"现象 |
| 训练稳定性 | 无 NaN / 损失尖峰，显存峰值 < 24 GB |

## 二、实验计划（分阶段推进）

> **数据集要点**（详见 `data/README.md`）：5k 用于试训与对照、20k 用于正式训练，两者按题面 SHA-256 互不重复；样本已含 `<think>` 推理轨迹与 `\boxed{}` 答案；单条长度 136–223 token（100% 覆盖于 1024 以内）。

### 阶段 0：准备【已完成 ✅】

- 环境：transformers 5.20 / peft 0.21 / trl 1.15 / bitsandbytes 0.50 / flash-attn 2.8.3（GPU 冒烟测试通过）；
- 模型：`/root/autodl-tmp/models/Qwen3.5-9B`（9.65B 参数，文件完整性校验通过）；
- 数据：`/root/autodl-tmp/data/`（SHA-256 与 `validation_report.json` 比对一致）。

### 阶段 1：冒烟测试【已完成 ✅ 2026-10-16】（300 条 × 60 step）

结果：loss 1.32 → 0.008（3.8 分钟）｜峰值显存 8.49 GiB｜可训练参数 43.3M (0.78%)｜3/3 未训样本格式合规且答案一致。详见「阶段 1 结果」。

- [x] assistant-only loss：已确认——63.1% token 参与训练，system/user 部分被正确屏蔽；
- [x] 序列构造正确：EOS（`<|im_end|>`）位于损失掩码内，生成正常收尾；
- [x] 显存峰值实测：8.49 GiB（目标 < 22 GB，余量充足）；
- [x] loss 平滑下降、无 NaN；抽样生成格式与答案全部通过。

### 阶段 2：5k 完整训练 + 对照【已完成 ✅ 2026-10-16】

- [x] 5k 全量训练 2 epochs（626 步 / 20.5 分钟 / 峰值显存 8.50 GiB / loss 1.246 → 0.0057），适配器：`qlora/outputs/5k/adapter`；
- [x] 与基座对照（24 条未训练样本）：格式合规 0% → **100%**，答案一致 0% → **75%**，详见「阶段 2 结果」；
- [ ] 人工抽查 50–100 条生成结果（对错、步骤质量、格式）——待补。

### 阶段 3：20k 扩展训练【已完成 ✅ 2026-10-16】

- [x] 20k 全量训练 2 epochs（2500 步 / 81.8 分钟 / loss 1.391 → 0.0051），适配器：`qlora/outputs/20k/adapter`；
- [x] 与 5k-LoRA 同集对比：隔离集 300 题**完全持平（37.7% vs 37.7%，逐题翻转 18↔18 净零）**——同分布数据量 4 倍未带来 OOD 泛化增益，详见「阶段 3 结果」；
- [x] 模板过拟合检查：确认 9 类 0% 题型需「模板族扩充 + 难度 3 数据 + 验算步骤样本」，已写入阶段 4 方向。

### 阶段 4：评测与迭代

- [x] 隔离评测集已就位：`data/high_math_eval_300/`（300 题，isolated_v2）与 `data/sft_v3/high_math_eval_v3/`（300 题，eval_v3）；
- [x] 评测工具链就绪：`\boxed{}` 提取 + math_verify LaTeX 等价判定 + 分题型/难度统计（`validate.py` / `rescore.py`）；
- [x] ① 数据扩充首轮（v3）完成：sft_v3 训练 + 双基准评测——**eval_v3 100%**｜iso300 34.3%（旧强项有回退）→ 详见「v3 实验结果」；
- [ ] ② **可选后续 v3.1**：B/C 新结构 14k + 全量旧 20k 均衡回放，共 34k；研究目标从“学习 QLoRA”转为“分布平衡与抗回退”；
- [ ] ③ 可选扩展：GRPO 可验证奖励、人工题与错题回流。

### 训练配置（阶段 1 实测版）

| 项目 | 取值 | 说明 |
|------|------|------|
| 量化 | 4-bit NF4 + double quant，bf16 计算 | QLoRA 标准配置 |
| LoRA | r=16，α=32，dropout=0.05 | target 覆盖 12 类线性层：全注意力 q/k/v/o + 线性注意力 in_proj_qkv/z/b/a、out_proj + MLP gate/up/down（实测模型为 8 层全注意力 + 24 层线性注意力混合架构） |
| max_length | 1024（实测最长 223/275，100% 覆盖） | 追求吞吐可降至 512 |
| 批大小 | per-device 8 × 梯度累积 2（等效 16） | 实测显存余量大，可加大 |
| 优化器 | paged_adamw_8bit，lr=2e-4，cosine + 3% warmup | QLoRA 常用配置 |
| 注意力实现 | flash_attention_2 + flash-linear-attention | 有效加速混合注意力层 |
| 梯度检查点 | 开启 | 以少量速度换显存 |
| 精度 | bf16 | 4090D 原生支持 |
| Loss | assistant_only_loss=True | 只对 assistant 回复（含 think 块）计损失；`<|im_end|>` 在掩码内 |

### 风险与对策

1. **数据为程序化合成**（规则校验，非逐条人工审校）→ 正式长跑前随机抽查 ≥100 条；
2. **模板同构导致"背模板"** → 评测必须模板隔离，必须加入变体题面（隔离集已暴露该风险，v3 已部分解决，仍需跨族评测）；
3. **分布偏移与能力回退** → v3、20k 均从同一基座独立训练，并非连续续训；v3 的 B/C 数据集中补强弱项，使部分旧强项权重下降；
4. **显存管理** → 峰值 8.5–8.7 GiB，余量充足。

### Notebook 推进路线

1. 实验目的与计划（本部分）
2. 环境与路径预检
3. 数据加载与 chat template 处理（assistant-only loss mask 验证）
4. QLoRA 配置与模型加载（4-bit + LoRA 注入）
5. 冒烟测试训练（300 条）✅
6. 5k / 20k 完整训练（5k ✅ 20k ✅）
7. 推理与隔离评测（iso300 三模型对比 ✅ / eval_v3 ✅）

## 📊 阶段 1：冒烟测试结果（2026-10-16 通过 ✅）

**配置**：5k 文件前 300 条 × 60 步 ｜ QLoRA 4bit NF4 + bf16 ｜ batch 8 × 梯度累积 2 ｜ lr 2e-4（cosine，warmup 10 步）｜ assistant-only loss ｜ flash-attn + flash-linear-attention

### 训练指标

| 指标 | 实测 | 验收目标 | 状态 |
|------|------|---------|------|
| Loss | 1.32 → 0.008（平滑，无 NaN） | 平滑下降 | ✅ |
| 峰值显存 | 8.49 GiB | < 22 GiB | ✅（余量充足） |
| 训练耗时 | 3.8 分钟（3.84 s/step） | — | ⚡ 5k ≈ 20 分钟/epoch |
| 可训练参数 | 43.3 M / 5.54 B = 0.78% | — | ✅ |
| 有效训练 token | ≈ 16.4 万（仅 assistant 段计入） | — | ✅ |

### 生成抽查（3 条未参与训练的样本，贪心解码）

| # | 题型 | 推理格式 | 答案校验 |
|---|------|---------|---------|
| 1 | 数列有理式极限 | think ✅ / boxed ✅ / EOS ✅ | ✅ 与参考答案一致（`5`） |
| 2 | 等比级数求和 | think ✅ / boxed ✅ / EOS ✅ | ✅ 一致（`-\frac{9}{2}`） |
| 3 | 一阶线性微分方程 | think ✅ / boxed ✅ / EOS ✅ | ✅ 一致（`y=\frac{37}{3}e^{-6x}-\frac{4}{3}`） |

### 关键过程发现

- 模型为**混合注意力架构**：8 层全注意力（q/k/v/o_proj）+ 24 层线性注意力（in_proj_qkv/z/b/a、out_proj），LoRA 目标层已相应覆盖 12 类；
- TRL 自带与模型原生模板逐行对应的训练模板（`qwen3_5_think_training.jinja`，含 `{% generation %}` 标记），`assistant_only_loss=True` 开箱即用，且 `<|im_end|>` 在损失掩码内；
- 已安装 `flash-linear-attention` 加速线性注意力层；`causal_conv1d` 缺失（回退慢速版，当前速度可接受）。

### 产物

- 冒烟适配器：`qlora/outputs/smoke/adapter`（含 `train_summary.json` 与 TensorBoard 日志）
- 训练 / 推理代码：`qlora/train.py`、`qlora/infer.py`（运行：`python train.py --mode smoke|5k|20k|v3`）
- 适配器「保存 → 重新加载 → 生成」闭环验证通过 ✅

**结论：训练链路全部打通，可进入阶段 2（5k 全量训练 + 基座对照）。**

## 📊 阶段 2：5k 训练与对照结果（2026-10-16）

### 训练指标（5k 全量 × 2 epochs）

| 指标 | 实测 |
|------|------|
| 步数 / 耗时 | 626 步 / 20.5 分钟（1.96 s/step） |
| Loss | 1.246 → 0.0057（全期均值 0.040） |
| 峰值显存 | 8.50 GiB |
| 可训练参数 | 43.3 M / 5.54 B（0.78%） |
| 产物 | `qlora/outputs/5k/adapter`（含 checkpoint-600/626 与 TensorBoard 日志） |

### 基座 vs 5k-LoRA 对照（24 条未训练样本①，贪心解码，严格协议）

| 指标 | 基座 | 5k-LoRA |
|------|------|---------|
| 格式合规率（think+boxed+EOS） | 0.0% | **100.0%** |
| 答案一致率（`\boxed{}` 提取 + 严格文本匹配） | 0.0% | **75.0%（18/24）** |
| 平均输出长度 / 单条耗时 | 1518 字符 / 26 s | 175 字符 / 9.3 s |

① 取样自 20k 文件（与 5k 训练集按题面 SHA-256 互斥）；正式隔离评测集留待阶段 4。

**基座说明**：基座输出为英文长篇「thinking process」（平均 1518 字符），24 条中 **0 条**能在 512 token 内完成 think 区块并给出最终答案 → 0% 反映的是**输出协议不合规**，而非解题能力为零。这正体现本阶段目标：把能力对齐到规范的中文解题协议。

**LoRA 未命中 6 条均为真实计算错误**（已逐条核对，非格式差异）：

| # | 题型 | 生成答案 | 参考答案 |
|---|------|---------|---------|
| 7 | derivative_polynomial_at_point | -82 | -86 |
| 9 | integral_symmetry | -14928/5 | -15072 |
| 17 | tangent_line_polynomial | y-50=21(x-4) | y-58=21(x-4) |
| 19 | integral_symmetry | 15504/5 | 16548 |
| 20 | total_differential | -10dx+54dy | -14dx+54dy |
| 23 | integral_symmetry | 404/15 | 476/15 |

错误集中在 `integral_symmetry`（对称性定积分 ×3）与个别系数/算术滑移 → 预期 20k 训练与阶段 4（LaTeX 等价判定 + 错题回流 + GRPO 可验算奖励）可进一步压低。

### 结论与下一步

- ✅ 阶段 2 目标达成：未见题上 100% 格式合规 + 75% 严格答案一致（基座同协议 0%）；
- ✅ 泛化初步验证：24 条覆盖 15 类题型，格式全部保持，无「背模板」崩溃迹象；
- 下一阶段：按计划进入 **阶段 3（20k 全量训练）**；或先补阶段 4 的隔离评测集与 LaTeX 等价判定后再决策。

完整生成文本：`qlora/outputs/compare/base_vs_5k_n24.json`

## 📊 隔离验证集首评（300 题，2026-10-16）

> 评测集：`data/high_math_eval_300/`（用户提供）——29 类题型各 10–11 题，题面与 5k/20k 训练集 SHA-256 零重合，独立模板族（isolated_v2），难度 2/3（训练数据仅含难度 1–2）。

### 总体指标（5k-LoRA，贪心解码，max_new_tokens=1024）

| 指标 | 结果 |
|------|------|
| 格式合规率 | **100%**（300/300，0 截断） |
| 等价正确率（math_verify LaTeX 等价判定） | **37.7%（113/300）** |
| 严格文本匹配 | 31.0%（等价判定救回 20 条等价写法，如 `x^5` vs `x^{5}`、`9e^{8x}(…)` vs `e^{8x}(9…)`） |
| 难度 2 正确率 | 48.9%（92/188） |
| 难度 3 正确率 | **18.8%（21/112）** |

### 分题型：明显的两极分化

- ✅ **满分/近满分**：数列有理极限 11/11、单调区间 11/11、曲线弧长 10/10、收敛半径 10/10、代换积分 9/10、二阶常系数 ODE 8/10、不定积分 8/10；
- ❌ **零分**（8 类)：隐函数求导、指数三角求导、方向导数、对称性积分、二重积分、偏导数、一阶线性 ODE、切平面。

### 失败性质（逐条抽查完整生成）

- 难度 2 失败：仍以**计算滑移**为主（偏导代点 `-5` 算成 `-1`）；
- 难度 3 失败：出现**方法性错误**（如隐函数求导把 `6xy` 的导数写成 `6y(y'+1)`）→ 对新结构/更高难度的深层泛化不足；
- 所有失败均与格式无关——**协议能力在 OOD 数据上零退化**（300/300 合规）。

### 结论与下一步

1. 5k 模型在**同族题面**（难度 1–2）≈75%，在**隔离新模板族 + 更难**上为 37.7% → 印证了「评测必须模板隔离」的必要性，当前模型对难度 3 与部分题型的泛化存在明确缺口；
2. 优先级：① 阶段 3 跑 20k 作对照（预期有限提升，但必须验证）；② 阶段 4 用 GRPO（可验算奖励）直接压制计算错误；③ 数据侧补充难度 3 与模板多样性样本；
3. 评测基建已就绪：`validate.py`（断点续跑 + 分题型统计）、`rescore.py`（等价重评分）、`diag_eval.py`（难度诊断）。

产物：`qlora/outputs/validate/5k_on_eval300.{samples,rescored}.jsonl`、`.summary.json`

## 📊 阶段 3：20k 训练与对比结果（2026-10-16）

### 训练指标（20k 全量 × 2 epochs）

| 指标 | 5k | 20k |
|------|-----|-----|
| 步数 / 耗时 | 626 步 / 20.5 min | 2500 步 / 81.8 min |
| Loss | 1.246 → 0.0057 | 1.391 → 0.0051 |
| 峰值显存 | 8.50 GiB | 8.50 GiB |
| 可训练参数 | 43.3 M（0.78%） | 43.3 M（0.78%） |

### 隔离集（300 题）对比：数据 4 倍 ≠ 泛化提升 ⚠️

| 指标 | 5k-LoRA | 20k-LoRA | 结论 |
|------|---------|----------|------|
| 格式合规率 | 100% | 100% | 协议学习稳定迁移 |
| **等价正确率** | **37.7%（113/300）** | **37.7%（113/300）** | **完全一致** |
| 难度 2 | 48.9% | 50.0% | ±1.1pp（噪声内） |
| 难度 3 | 18.8% | 17.0% | ±1.8pp（噪声内） |
| 逐题翻转 | — | 18 条改善 / 18 条变差 | **净变化 0** |

分题型变化（每类 n=10–11，±2 属二项噪声）：`integral_by_parts` −4、`improper_integral` −2、`ode_exponential_growth` +2、`ode_first_order` +2、`tangent_line` +2 等——**正负相抵，无显著差异**。

### 结论（重要负结果）

1. **同分布数据量提升（5k→20k）对模板隔离集泛化零增益**。瓶颈不在数据量，而在：
   - ① 模板族多样性：5k/20k 同为 29 类模式，20k 只是同模板的更多参数组合；
   - ② 难度覆盖：训练数据仅难度 1–2，评测含难度 3；
   - ③ 推理时的计算可靠性（退化解法之外的算术滑移）。
2. 协议能力（100% 合规）与分布内解题模式全部稳定，证实微调本身的"格式对齐"目标达成；
3. **阶段 4 方向已明确**：
   - 数据侧：新增模板族、难度 3 样本、含显式验算步骤的样本（重点针对 9 类 0% 题型）；
   - 算法侧：GRPO（`canonical_answer` + math_verify 作可验证奖励）直接优化正确率；
   - 评测侧：固定隔离集，逐版本对比（5k / 20k / 后续版本）。

产物：`qlora/outputs/20k/adapter`；对照汇总：`qlora/outputs/validate/20k_on_eval300.summary.json`

## 六、v3 实验结果：定向增强与能力重分配（2026-10-16）

### 6.1 数据、训练与实验关系

- **数据**：`data/sft_v3/`，共 20,000 条，由 A 旧分布均衡复用 6k、B 新结构 9.5k、C 难度 3 多步题 4.5k 组成；全部含分步推导和显式验算段。
- **训练**：2 epochs，2500 步，93.05 分钟，loss 1.7548 → 0.0110，峰值显存 8.74 GiB。
- **适配器**：`qlora/outputs/v3/adapter`。
- **重要口径**：5k、20k、v3 都是从同一基座分别训练出的独立 LoRA adapter；v3 不是在 20k adapter 上继续训练，因此后文使用“能力回退/能力重分配”，不把它直接定性为经典连续学习中的灾难性遗忘。

### 6.2 同源结构隔离集 eval_v3

| 模型 | 格式合规 | 严格匹配 | 等价正确率 | 相对 20k |
|------|---------:|---------:|-----------:|-----------:|
| 5k-LoRA | 100% | 88.0%（264/300） | 94.3%（283/300） | +1.0pp |
| 20k-LoRA | 100% | 86.7%（260/300） | 93.3%（280/300） | 基线 |
| **v3-LoRA** | **100%** | **99.3%（298/300）** | **100.0%（300/300）** | **+6.7pp** |

v3 的两条非严格匹配均为正确的等价直线方程，math_verify 判定合理。20 个重点题型在 v3 模型下均为 15/15。这个结果证明定向数据消除了 eval_v3 中旧适配器剩余的 17–20 个错误，但**不是从约 35% 跃升到 100%**：5k 和 20k 在该集合上原本已经达到 94.3% 和 93.3%。

### 6.3 历史跨结构基准 iso300

| 指标 | 5k-LoRA | 20k-LoRA | v3-LoRA |
|------|---------:|----------:|--------:|
| 总等价正确率 | 37.7%（113/300） | 37.7%（113/300） | **34.3%（103/300）** |
| 难度 2 | 48.9% | 50.0% | 47.3% |
| 难度 3 | 18.8% | 17.0% | 12.5% |

**类别间能力重新分配**（v3 vs 20k，仅列变化较大的类别）：

- 提升：`integral_by_parts` 0→8、`improper_integral` 5→10、`limit_factor` 0→2；
- 回退：`integral_substitution` 10→1、`quadratic_multivariable_minimum` 8→3、`ode_second_order` 7→2。

### 6.4 结果解释

1. **定向数据有效，但增益范围有限**：v3 把同源 eval_v3 从约 93%–94%提高到 100%，说明训练结构与显式验算能够修复目标分布中的剩余错误。
2. **eval_v3 存在明显天花板效应**：旧适配器已经超过 93%，且评测题与训练题来自同一生成体系、语言风格相近、全部为难度 2；因此它适合验证受控结构迁移，不足以单独证明开放世界泛化。
3. **旧基准回退来自训练分布变化**：A 组本身在 29 类中近似均衡，每类约 206–207 条；真正的偏移来自 B/C 共 14k 集中补强少数弱项，使未进入 B/C 的旧强项在总训练 token 中权重下降。
4. **低训练 loss 不等于全面能力提升**：v3 loss 降至 0.011，反映模型高度适应 v3 分布；iso300 的回退表明这种拟合并未自动转化为更广泛的跨模板能力。

### 6.5 可选后续研究（不属于本次 QLoRA 学习的必做项）

- **v3.1 数据方案**：B/C 新结构 14k + 全量旧 20k，共 **34k**，而不是 40k；用双基准联合早停研究如何平衡新旧能力。
- **建议验收**：eval_v3 ≥ 95%、iso300 ≥ 40%，并明确“难度 3 ≥ 25%”指 iso300 难度 3 正确率。
- **更强外部效度**：增加人工编写或另一生成器产生的跨族评测，再决定是否进入 GRPO。

证据文件：`qlora/outputs/v3/train_summary.json`、`qlora/outputs/validate/*eval_v3_300*`、`qlora/outputs/validate/*eval300*`；对比脚本：`qlora/compare_v3.py`。

## 七、实验总结与最终结论

### 7.1 研究问题回答

| 研究问题 | 实验证据 | 结论 |
|---------|---------|------|
| 单卡 4090D 能否完成 9B 模型 QLoRA？ | 5k/20k/v3 均稳定完成，峰值显存 8.50–8.74 GiB | **可以，且显存余量充足** |
| QLoRA 能否学会中文数学解题协议？ | 所有正式适配器在两个 300 题集合上格式合规率均为 100% | **可以，格式对齐非常稳定** |
| 同构数据从 5k 扩大到 20k 是否提升泛化？ | iso300 均为 113/300（37.7%） | **没有；数据量不是主要瓶颈** |
| 新结构、难度 3 和显式验算是否有效？ | eval_v3：20k 93.3% → v3 100% | **对目标受控分布有效** |
| 定向增强是否自动改善跨结构能力？ | iso300：20k 37.7% → v3 34.3% | **不会，可能产生能力重分配和旧分布回退** |

### 7.2 对 QLoRA 的核心认识

1. **QLoRA 解决的是训练资源问题，不自动解决数据与泛化问题。** 4-bit 量化和低秩更新显著降低了显存需求，但模型学到什么仍由训练分布决定。
2. **LoRA adapter 是共享行为偏置，不是按题型隔离的知识仓库。** 同一组低秩参数同时服务所有题型；针对少数题型的梯度更新可能改变其他题型的解题倾向。
3. **训练 loss 只能说明对训练分布的拟合程度。** 20k 和 v3 的末期 loss 都很低，但跨结构评测没有同步提高。
4. **评测集设计决定结论可信度。** 精确题面零重合只是最低要求；还要区分同源生成器、结构签名、难度、语言风格和真正的跨族测试。
5. **负结果同样有价值。** 5k→20k 无增益，以及 v3 在新集合满分但旧集合回退，共同说明“更多数据”和“目标集合满分”都不能替代分布分析。

### 7.3 对初始成功标准的复盘

- **格式合规率 ≥95%：已达到。** 正式评测均为 100%。
- **训练稳定、显存 <24GB：已达到。** 无 NaN，峰值最高 8.74 GiB。
- **正确率相对基座提升 ≥15pp：局部证据支持，但未在 iso300 上完成严格的基座同配置对照。** 24 条早期对照受到基座输出协议不合规影响，不能等价为纯数学能力差异。
- **泛化无明显退化：未完全达到。** v3 在同源 eval_v3 提升，但 iso300 回落且难度 3 只有 12.5%。

### 7.4 最终结论

本实验已经完成学习 QLoRA 所需的完整闭环：4-bit 模型加载、LoRA 模块选择、assistant-only loss 验证、冒烟测试、正式训练、adapter 保存与加载、隔离评测、等价答案重评分、对照实验和失败分析。工程目标与格式对齐目标已充分达成；同时，实验明确揭示了 QLoRA 的边界——参数高效微调能够低成本地塑造目标行为，却不能绕过数据分布、评测外部效度和能力权衡问题。

因此，本项目可以作为“QLoRA 学习实验”正式结束。34k 均衡回放、人工跨族评测和 GRPO 应被定义为新的研究课题，而不是当前实验尚未完成的尾项。

### 7.5 证据与复现索引

- 训练代码：`qlora/train.py`；统一配置：`qlora/config.py`；共享加载逻辑：`qlora/common.py`。
- 训练摘要：`qlora/outputs/{smoke,5k,20k,v3}/train_summary.json`。
- 适配器与 checkpoint：`qlora/outputs/{5k,20k,v3}/`。
- 双基准样本和统计：`qlora/outputs/validate/`。
- v3 对比脚本：`qlora/compare_v3.py`；数据质检：`qlora/qa_v3.py`。
- **评分口径提醒**：`5k_on_eval300.summary.json` 保留了首次 31.0% 的旧评分；37.7% 来自 `5k_on_eval300.rescored.jsonl` 中 math_verify 重评分后的 113/300。最终汇报应明确使用重评分口径，避免两份数字混用。


## 八、实验启发与实际应用

### 8.1 微调最擅长塑造行为，不等于保证所有答案正确

本实验中最稳定的提升是输出协议：中文分步推导、`<think>`、显式验算、`\boxed{}` 和正常结束。在两个 300 题评测集上，所有正式适配器的格式合规率都达到 100%。这说明 QLoRA 特别适合学习固定工作流、表达风格、分类规则、工具调用顺序和结构化输出。

但 iso300 只有约 34%–38%，说明“能按要求回答”和“面对任意新问题都能答对”是两个不同目标。SFT 可以显著提高行为一致性，却不能单独保证复杂推理和精确计算的可靠性。

### 8.2 数据结构比数据数量更重要

5k 与 20k 在 iso300 上同为 113/300，说明把相同模板换数字、换系数后扩充四倍，并没有带来跨结构泛化。新增样本是否有价值，应优先判断它是否包含新的决策结构、失败模式和边界情况，而不只是参数变化。

> 高质量的结构多样性，通常比大量同构改写更重要。

### 8.3 训练 loss 不能替代真实评测

20k 和 v3 的最终 loss 分别低至 0.0051 与 0.0110，但 iso300 正确率没有同步提升。低 loss 只能证明模型适应了当前训练分布，不能证明它已经具备跨模板、跨来源和更高难度的泛化能力。模型选择应依据独立回归集，而不是依据最低训练 loss。

### 8.4 题面零重合只是评测可信度的第一层

eval_v3 与训练集在题面哈希和结构签名上均为零重合，但旧适配器已经达到 93%–94%。这说明同一生成器、相似语言风格、相近参数范围和统一答案协议仍会使评测偏容易。更可靠的评测应形成三层：

1. **同源参数隔离**：检查是否记住具体题面；
2. **同源结构隔离**：检查是否能迁移到相邻结构；
3. **跨来源或人工评测**：检查真实业务泛化能力。

eval_v3 主要属于第二层；后续若追求实际解题能力，应增加人工题或另一生成器产生的跨族题目。

### 8.5 定向微调会重新分配能力

v3 在目标分布上达到 100%，同时在 iso300 上从 37.7%回落到 34.3%。这说明模型能力不是单一分数，而是多个题型和难度上的分布。LoRA 参数由所有题型共享，集中强化少数结构时，部分旧任务的行为倾向可能下降。

因此，每次微调都应同时评估：新增能力是否提高、旧能力是否下降、下降的能力是否属于关键业务。生产系统需要固定回归集，而不能只报告目标集合上的最好成绩。

### 8.6 不一定需要一个 adapter 解决所有任务

5k、20k 和 v3 是从同一基座独立训练的不同 adapter，各自形成了不同的能力偏置。实际部署中可以保留多个专用 adapter，通过任务路由、服务端点切换或 adapter 融合分别服务通用数学、难题推导和特定业务流程，而不必无限把所有数据混入一个 adapter。

### 8.7 精确任务需要模型、工具和验证器协作

iso300 中不少错误属于代点、符号、分数化简和中间项遗漏。模型可能知道方法，但自回归生成不是确定性的符号计算器。可靠数学系统更适合采用：

$$\text{模型规划与解释} + \text{计算工具} + \text{结果验证器}$$

模型负责识别题型和组织步骤，SymPy 或计算器负责精确运算，验证器检查最终答案；不一致时再触发重新推导。

### 8.8 微调、RAG 与工具的分工

| 技术 | 最适合解决的问题 | 本实验对应启发 |
|------|------------------|---------------|
| 微调 / QLoRA | 稳定行为、输出格式、专业语气、固定工作流、工具调用习惯 | 100% 格式合规证明行为对齐有效 |
| RAG | 经常更新、需要引用、来自内部文档或受权限控制的知识 | 不应依靠微调记忆变化的知识与规则 |
| 计算工具 / 验证器 | 数值计算、符号运算、SQL、单位换算和确定性规则 | 用于解决模型的算术滑移和结果复核 |

实际系统通常应组合三者：

$$\text{RAG 提供依据} \rightarrow \text{微调模型按流程处理} \rightarrow \text{工具精确执行} \rightarrow \text{验证器检查结果}$$

### 8.9 对后续项目的总体启发

在开始微调前，应先定义真实部署分布和固定回归集，再围绕错误类型设计数据。训练完成后，应同时报告目标分布、历史分布和跨来源分布，而不是只看 loss 或单一高分。5k→20k 无增益以及 v3 的新旧能力权衡并非失败，而是排除了“更多同构数据必然提升泛化”和“目标集合满分代表模型整体更强”这两个错误假设。

> **最终启发**：QLoRA 能以很低的资源成本稳定塑造模型行为，但模型最终形成什么能力，主要取决于数据分布；要构建可靠系统，还必须结合独立评测、RAG、计算工具和结果验证。
